# Лабораторная 02 — Аугментация данных и устойчивость

По курсу ASR

Выполнила Пелсе Александра (506841), магистратура 2 курс

1. Создайте окружение venv-asr-lab02 и установите `requirements.txt`. 
    
    ВЫПОЛНЕНО:

    >deactivate

    >python -m venv venv-asr-lab02

    >.\venv-asr-lab02\Scripts\Activate.ps1

    >python -m pip install --upgrade pip

    >pip install -r requirements.txt

2. Проверьте данные: `python scripts/make_lab02_subset.py`. 
    
    ВЫПОЛНЕНО:
    
    Этот скрипт проверил целостность данных.
3. Запустите `python scripts/run_lab_02.py --skip-asr`, чтобы сгенерировать детерминированно аугментированные аудио и графики. 
    
    ВЫПОЛНЕНО:

    - запуск скрипта "augment_audio.py"

    * add_noise_at_snr() - добавление белого шума. Параметр snr_db - это соотношение сигнал/шум (Signal-to-Noise Ratio). Чем оно ниже, тем громче шум. Подставляется из conditions, для значения ("noise_0db", {"snr_db": 0.0}) шум по мощности равен голосу.
    * make_synthetic_rir() - добавление реверберации.
    * apply_speed_perturbation() - изменение скорости, замедление 0.9x и ускорение 1.1x. Используется полифазное пересемплирование (signal.resample_poly). Оно не просто выкидывает сэмплы, а математически пересчитывает сетку частоты, чтобы изменить темп речи без появления цифровых артефактов.

    В результате работы скрипта для каждого аудио из манифеста создаются:
	1. clean — чистый звук (оригинал)
	2. noise_20db — легкий шум
	3. noise_10db — средний шум
	4. noise_0db — сильный шум
	5. reverb_room_a — эхо в комнате А
	6. speed_0.9x — замедленная речь
	7. speed_1.1x — ускоренная речь

    - скрипт "plot_augmentation_features.py"

    В результате работы скрипта сгенерировано два файла clean_vs_augmented.png и specaugment_feature_only.png

4. Изучите `clean_vs_augmented.png` и `specaugment_feature_only.png`. Объясните маски по времени и частоте.

    ОТВЕТ:

    На визуализации clean_vs_augmented.png маскирования нет. Здесь показаны естественные искажения; сравнивается чистый звук и звук, в который добавили шум (из файла 1272-128104-0000__noise_10db.wav). Шум размазан по всему графику, а не вырезан полосами, как при маскировании.

    На изображении specaugment_feature_only.png представлено искусственное (математическое) маскирование признаков (SpecAugment). Скрипт берет чистую спектрограмму и затирает на ней центральные полосы по времени и частоте. Если обратиться к функции specaugment(), то параметр *time_width: int = 24* указывает ширину вертикальной полосы (маски времени). Внутри функции эта полоса зануляется:
    
    ```
    result[:, time_start:time_start + time_width] = matrix.min()
    ```

    Параметр frequency_width: int = 10 говорит о том, что горизонтальная маска сотрет ровно 10 признаков (каналов частот) из 80:

    ```
    result[frequency_start:frequency_start + frequency_width, :] = matrix.min()
    ```
    
5. Запустите `python scripts/run_lab_02.py --allow-model-download`, чтобы скачать и закэшировать `tiny.en`. Для последующих/автономных запусков запускайте без этого флага (или передайте `--model-dir /path/to/local/tiny.en`).

    ВЫПОЛНЕНО:
    
    в результате работы скрипта 

    * была загружена модель Whisper Tiny (tiny.en) из репозитория Hugging Face и сохранена в локальный кэш.
    * модель выполнила инференс (распознавание речи) для всех сгенерированных аудиофайлов .wav во всех семи акустических условиях:

        - извлечение признаков: преобразование .wav сначала в Лог-Мел спектрограммы (матрицы из 80 признаков на кадр).
        - подача матриц на вход нейросети (архитектура модели - трансформер)
        - модель предсказывает аиболее вероятную последовательность текстовых символов и слов

    * была посчитана метрика WER*
    * были сгенерированы отчеты per_item_results.csv (детализированные по каждому файлу) и aggregate_results.csv (сгруппированные по типу искажения).

    Ликбез: 

    *WER (Word Error Rate) — это ключевая метрика качества систем распознавания речи, которая показывает коэффициент ошибок на уровне слов. Она рассчитывается как отношение суммы замененных (S), пропущенных (D) и вставленных лишних слов (I) к общему количеству слов в эталонном тексте (N). Чем ниже значение WER, тем точнее модель распознала речь (WER = 0% означает идеальное совпадение).


6. Прочитайте `per_item_results.csv` и `aggregate_results.csv`.

In [1]:
import pandas as pd
from pathlib import Path

# Определяем путь к папке с результатами
# (Используем Path, чтобы код работал и на Windows, и на Linux/Mac)
results_dir = Path("outputs/results")

# 1. Читаем и отображаем первые 7 строк детального отчета
per_item_df = pd.read_csv(results_dir / "per_item_results.csv")
print("=== Первые 7 строк детального отчета (per_item_results) ===")
display(per_item_df.head(7))  # Функция display() делает таблицу красивой в Jupyter

# 2. Читаем и отображаем весь агрегированный отчет
aggregate_df = pd.read_csv(results_dir / "aggregate_results.csv")
print("\n=== Агрегированные результаты (aggregate_results) ===")
display(aggregate_df)

=== Первые 7 строк детального отчета (per_item_results) ===


,item_id,condition,parameters,seed,reference_raw,reference_normalized,hypothesis_raw,hypothesis_normalized,wer,cer,audio_seconds,decode_seconds,rtf
0,1272-128104-0000,clean,{},17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,5.855000,0.586521,0.100174
1,1272-128104-0000,noise_20db,"{""snr_db"": 20.0}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,5.855000,0.505364,0.086313
2,1272-128104-0000,noise_10db,"{""snr_db"": 10.0}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,5.855000,0.555608,0.094895
3,1272-128104-0000,noise_0db,"{""snr_db"": 0.0}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Closer is the apostle of the middle classe...,mr closer is the apostle of the middle classes...,0.470588,0.292135,5.855000,0.481250,0.082195
4,1272-128104-0000,reverb_room_a,"{""rir"": ""room_a""}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,5.855000,0.503508,0.085996
5,1272-128104-0000,speed_0.9x,"{""speed_factor"": 0.9}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,6.505562,0.484424,0.074463
6,1272-128104-0000,speed_1.1x,"{""speed_factor"": 1.1}",17,MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CL...,mister quilter is the apostle of the middle cl...,Mr. Quilter is the apostle of the middle class...,mr quilter is the apostle of the middle classe...,0.058824,0.044944,5.322750,0.604576,0.113583



=== Агрегированные результаты (aggregate_results) ===


,condition,utterance_count,reference_word_count,mean_wer,mean_cer,mean_rtf
0,clean,5,82,0.103958,0.059503,0.076993
1,noise_0db,5,82,0.654995,0.366894,0.131646
2,noise_10db,5,82,0.165625,0.090197,0.071162
3,noise_20db,5,82,0.128958,0.070029,0.071149
4,reverb_room_a,5,82,0.140625,0.070363,0.071918
5,speed_0.9x,5,82,0.083958,0.047959,0.065740
6,speed_1.1x,5,82,0.103958,0.059503,0.089515


ОТВЕТ:

Мы видим результаты стресс-тестирования архитектуры Whisper Tiny на устойчивость к различным типам акустических искажений.

1. Анализ устойчивости к шуму (Критическая зона)

* Идеальные условия (clean): Средний по выборке WER составляет ~10.4%. Это базовая точность (baseline) для модели Tiny.
* Умеренный шум (noise_20db и noise_10db): Модель держится стабильно. При добавлении шума в 10 дБ средний WER вырастает незначительно — до 16.5%. В детализированном выводе таблицы для item_id 1272-128104-0000 метрика WER вообще не изменилась (осталась 0.058), что говорит о высокой устойчивости Whisper к легкому белому шуму.
* Сильный шум (noise_0db): Происходит катастрофическая деградация (breakdown) качества. Средний WER поднимается в несколько раз до 65.5%. На примере детализированного вывода видим, что 

    "mister quilter..." -> модель выдала "mr closer..."
    "and we are glad to welcome his gospel" -> превратился в "and he has led to the ultimate whose gospel". 
    
    Когда мощность шума приближена к мощности речи (0 дБ), Whisper Tiny начинает галлюцинировать и подменять слова и фразы.

2. Анализ устойчивости к реверберации (reverb_room_a)

* Имитация пустой комнаты ухудшает распознавание лишь незначительно: средний WER поднимается с 10.4% до 14%. Модели на базе архитектуры Transformer (такие как Whisper) за счет механизма Self-Attention отлично умеют компенсировать временные размытия звука, вызванные реверберацией.

3. Анализ устойчивости к темпу речи (Парадокс скорости)

* Ускорение (speed_1.1x): WER осталась абсолютно идентичной базовой (clean) — 10.4%.
* Замедление (speed_0.9x): средний WER улучшился до 8.4% (это лучший результат во всей таблице).
	
    Почему это произошло: исходный датасет LibriSpeech содержит естественную быструю речь. Математическое замедление аудиосигнала дало декодеру модели чуть больше временных кадров для анализа каждого звука, что помогло точнее распознать слова.

4. Производительность системы (Метрика RTF)

RTF (Real-Time Factor) — это отношение времени декодирования к длительности аудио. Если RTF < 1.0, модель работает быстрее реального времени.

* Во всех условиях RTF колеблется в районе 0.06 – 0.13.
* Это означает, что на обработку 1 минуты аудио у модели уходит всего около 4–8 секунд.
* Нюанс: При сильном шуме (noise_0db) средний RTF почти удваивается (0.131). Модель тратит больше времени и вычислительных ресурсов на генерацию текста, так как сильно «сомневается» в вероятностях токенов из-за грязного сигнала.

Вывод: 
    
    Архитектура Whisper Tiny демонстрирует высокую устойчивость к изменению темпа речи (показав лучший WER = 8.4% при замедлении 0.9x) и умеренной реверберации. Однако модель критически чувствительна к сильному зашумлению сигнала: при SNR 0 дБ средняя ошибка WER возрастает более чем в 6 раз (до 65.5%), переводя систему в режим галлюцинирования. А также RTF для сильно зашумленных аудио критически вырастает. Как вариант для бизнес-применения модели в шумных средах необходим предварительный этап цифрового шумоподавления (Denoising).